In [73]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from scipy.spatial.distance import cosine, pdist

In [74]:
CT_COL     = "TCRClonotype"
SAMPLE_COL = "Sample_Origin"
LY49C_COL  = "Ly-49C-Ly-49I-Klra3-Klra9-AMM2139-pAbO"

min_size = 5
plots_per_page = 16
thresholds = np.linspace(1.0, 0.00, 101)
default_q = 0.925

DEBUG_PROBS = False

In [75]:
markers_b10br = [
    "RiO-Allo:H-2Kb-ATLVFHNL-pAbO",
    "RiO-Allo:H-2Kb-EEEPVKKI-pAbO",
    "RiO-Allo:H-2Kb-HIYEFPQL-pAbO",
    "RiO-Allo:H-2Kb-INFDFPKL-pAbO",
    "RiO-Allo:H-2Kb-RAYLFNSV-pAbO",
    "RiO-Allo:H-2Kb-RTYTYEKL-pAbO",
    "RiO-Allo:H-2Kb-SNYLFTKL-pAbO",
    "RiO-Allo:H-2Kb-SSYTFPKM-pAbO",
    "RiO-Allo:H-2Kb-SVYVYKVL-pAbO",
    "RiO-Allo:H-2Kb-VAFDFTKV-pAbO",
    "RiO-Allo:H-2Kb-VGPRYTNL-pAbO",
    "RiO-Allo:H-2Kb-VIVRFLTV-pAbO",
    "RiO-Allo:H-2Kb-VSFTYRYL-pAbO",
]

peptides_b10br = [
    "ATLVFHNL","EEEPVKKI","HIYEFPQL","INFDFPKL","RAYLFNSV","RTYTYEKL",
    "SNYLFTKL","SSYTFPKM","SVYVYKVL","VAFDFTKV","VGPRYTNL","VIVRFLTV","VSFTYRYL"
]

markers_balbc = [
    "RiO-Allo:H-2Kb-ATLVFHNL-pAbO",
    "RiO-Allo:H-2Kb-HIYEFPQL-pAbO",
    "RiO-Allo:H-2Kb-INFDFPKL-pAbO",
    "RiO-Allo:H-2Kb-RAYLFNSV-pAbO",
    "RiO-Allo:H-2Kb-RTYTYEKL-pAbO",
    "RiO-Allo:H-2Kb-SNYLFTKL-pAbO",
    "RiO-Allo:H-2Kb-SSYTFPKM-pAbO",
    "RiO-Allo:H-2Kb-SVYVYKVL-pAbO",
    "RiO-Allo:H-2Kb-VAFDFTKV-pAbO",
    "RiO-Allo:H-2Kb-VGPRYTNL-pAbO",
    "RiO-Allo:H-2Kb-VIVRFLTV-pAbO",
    "RiO-Allo:H-2Kb-VSFTYRYL-pAbO",
    "RiO-H-2:H-2Kd-SYFPEITHI-ADEX5099-pAbO"
]

peptides_balbc = [
    "ATLVFHNL","HIYEFPQL","INFDFPKL","RAYLFNSV","RTYTYEKL","SNYLFTKL",
    "SSYTFPKM","SVYVYKVL","VAFDFTKV","VGPRYTNL","VIVRFLTV","VSFTYRYL","SYFPEITHI"
]

In [76]:
def filter_zero_dextramer_cells(
    df: pd.DataFrame,
    markers: list[str],
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Remove cells (rows) where the sum of all dextramer marker counts is zero.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with cells as rows
    markers : list[str]
        List of dextramer marker column names
    verbose : bool
        If True, print filtering statistics
        
    Returns
    -------
    pd.DataFrame
        Filtered dataframe with zero-dextramer cells removed
    """
    dex_sum = df[markers].sum(axis=1)
    mask = dex_sum > 0
    n_before = len(df)
    n_after = int(mask.sum())
    n_removed = n_before - n_after
    
    if verbose:
        pct_removed = 100.0 * n_removed / n_before if n_before > 0 else 0.0
        print(f"[filter_zero_dextramer] Removed {n_removed:,} / {n_before:,} cells "
              f"({pct_removed:.2f}%) with zero total dextramer counts")
    
    return df[mask].copy()

In [77]:
def row_normalise(
    X: np.ndarray,
    eps: float = 0.0,
    debug: bool = False,
    tag: str = "P",
) -> np.ndarray:
    """
    Row-normalise raw peptide intensities/counts -> probabilities per cell.

    If a row sums to 0 (all peptides 0), we return an all-zero probability row.
    Those contribute 0 entropy under our convention (see renyi_entropy).
    """
    X = np.asarray(X, dtype=float)
    rs = X.sum(axis=1, keepdims=True)

    zero = (rs.squeeze() <= 0)
    if debug:
        frac = 100.0 * float(np.mean(zero)) if X.shape[0] else 0.0
        print(f"[DEBUG] {tag}: {frac:.2f}% cells have zero RiO mass (all peptides 0).")

    rs_safe = rs.copy()
    rs_safe[rs_safe <= 0] = 1.0
    P = X / rs_safe

    if eps > 0:
        P = np.maximum(P, eps)
        P = P / P.sum(axis=1, keepdims=True)

    return P

In [78]:
def renyi_entropy(P: np.ndarray, alpha: float, axis: int = -1) -> np.ndarray:
    """
    Renyi entropy in bits for probability vectors P.

    IMPORTANT: Handles all-zero rows safely (returns 0 for those rows).
      alpha=1 -> Shannon
      alpha=0 -> Hartley (log2 support size)
    """
    P = np.asarray(P, dtype=float)
    s = P.sum(axis=axis)
    zero_mask = (s == 0)

    def apply_zero_mask(H):
        if np.isscalar(H):
            return 0.0 if zero_mask else H
        H = np.asarray(H, dtype=float)
        H[zero_mask] = 0.0
        return H

    if alpha == 1.0:
        with np.errstate(divide="ignore", invalid="ignore"):
            logP = np.where(P > 0, np.log2(P), 0.0)
        H = -(P * logP).sum(axis=axis)
        return apply_zero_mask(H)

    if alpha == 0.0:
        k = np.sum(P > 0, axis=axis)
        k = np.maximum(k, 1)
        H = np.log2(k)
        return apply_zero_mask(H)

    S = np.sum(np.power(np.maximum(P, 0.0), alpha), axis=axis)
    S = np.maximum(S, 1e-300)
    H = (1.0 / (1.0 - alpha)) * np.log2(S)
    return apply_zero_mask(H)

In [79]:
def mean_pairwise_cosine_similarity(P: np.ndarray) -> float:
    """
    P: row-normalised per cell (n_cells x n_peptides).
    Returns mean pairwise cosine similarity, or NaN if <2 cells.
    """
    if P.shape[0] < 2:
        return np.nan
    return float(np.mean(1.0 - pdist(P, metric="cosine")))

In [80]:
def compute_clonotype_summaries(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
    entropy_alpha: float = 1.0,
    debug_probs: bool = False,
) -> dict:
    """
    Returns dict ct -> summary:
      - mean_pattern: mean of per-cell normalised vectors
      - mean_entropy: mean Renyi entropy (alpha) across cells
      - mean_coherence: mean pairwise cosine similarity across cells
      - n_cells: number of cells
    """
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]
    out = {}

    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        P = row_normalise(X, debug=debug_probs, tag=f"P_all[{ct}]")

        ent = float(np.mean(renyi_entropy(P, alpha=float(entropy_alpha), axis=1)))
        coh = mean_pairwise_cosine_similarity(P)

        out[ct] = {
            "mean_pattern": P.mean(axis=0),
            "mean_entropy": ent,
            "mean_coherence": coh,
            "n_cells": int(P.shape[0]),
        }
    return out

In [81]:
def apply_ly49c_filter_one_sample(df_s: pd.DataFrame, ly49c_col: str, q: float) -> tuple[pd.DataFrame, float]:
    """
    Within ONE sample: keep Ly49C <= quantile(q).
    Returns (filtered_df, Ts).
    """
    if q >= 1.0:
        return df_s.copy(), float(df_s[ly49c_col].max())
    Ts = float(df_s[ly49c_col].quantile(q))
    return df_s[df_s[ly49c_col] <= Ts].copy(), Ts

In [82]:
def pattern_distance(p1: np.ndarray, p2: np.ndarray, metric: str) -> float:
    if metric == "cosine":
        if np.allclose(p1, 0) or np.allclose(p2, 0):
            return 1.0
        return float(cosine(p1, p2))
    if metric == "l1":
        return float(np.sum(np.abs(p1 - p2)))
    raise ValueError(f"Unknown metric: {metric}")

In [83]:
def ly49c_sweep_metrics_one_sample(
    df_s: pd.DataFrame,
    ct_col: str,
    ly49c_col: str,
    markers: list[str],
    thresholds: np.ndarray,
    min_size: int = 5,
    entropy_alpha: float = 1.0,
    debug_probs: bool = False,
) -> pd.DataFrame:
    """
    Computes sweep metrics for ONE sample (entropy order fixed).

    Outputs per q:
      - retention
      - median cosine dist between mean clonotype patterns (baseline vs filtered)
      - median L1 dist
      - mean entropy change (filtered - baseline) across common clonotypes
      - mean coherence change (filtered - baseline)
    """
    baseline = compute_clonotype_summaries(
        df_s, ct_col, markers, min_size=min_size, entropy_alpha=entropy_alpha, debug_probs=debug_probs
    )
    n0 = len(df_s)

    rows = []
    for q in thresholds:
        df_f, Ts = apply_ly49c_filter_one_sample(df_s, ly49c_col, float(q))
        filtered = compute_clonotype_summaries(
            df_f, ct_col, markers, min_size=min_size, entropy_alpha=entropy_alpha, debug_probs=debug_probs
        )

        common = set(baseline) & set(filtered)
        if common:
            cos_d = [pattern_distance(baseline[c]["mean_pattern"], filtered[c]["mean_pattern"], "cosine") for c in common]
            l1_d  = [pattern_distance(baseline[c]["mean_pattern"], filtered[c]["mean_pattern"], "l1") for c in common]
            ent_d = [filtered[c]["mean_entropy"] - baseline[c]["mean_entropy"] for c in common]
            coh_d = [filtered[c]["mean_coherence"] - baseline[c]["mean_coherence"] for c in common]

            row = {
                "percentile": float(q),
                "Ts_ly49c": float(Ts),
                "n_cells": int(len(df_f)),
                "pct_cells_retained": 100.0 * len(df_f) / n0 if n0 else np.nan,
                "n_clonotypes_baseline_ge5": int(len(baseline)),
                "n_clonotypes_filtered_ge5": int(len(filtered)),
                "n_common_clonotypes_ge5": int(len(common)),
                "median_cosine_dist": float(np.median(cos_d)),
                "median_l1_dist": float(np.median(l1_d)),
                "mean_entropy_change": float(np.mean(ent_d)),
                "mean_coherence_change": float(np.nanmean(coh_d)),
            }
        else:
            row = {
                "percentile": float(q),
                "Ts_ly49c": float(Ts),
                "n_cells": int(len(df_f)),
                "pct_cells_retained": 100.0 * len(df_f) / n0 if n0 else np.nan,
                "n_clonotypes_baseline_ge5": int(len(baseline)),
                "n_clonotypes_filtered_ge5": int(len(filtered)),
                "n_common_clonotypes_ge5": 0,
                "median_cosine_dist": np.nan,
                "median_l1_dist": np.nan,
                "mean_entropy_change": np.nan,
                "mean_coherence_change": np.nan,
            }

        rows.append(row)

    return pd.DataFrame(rows)

In [84]:
def ly49c_sweep_entropy_orders_one_sample(
    df_s: pd.DataFrame,
    sample_name: str,
    ct_col: str,
    ly49c_col: str,
    markers: list[str],
    thresholds: np.ndarray,
    orders: list[float],
    min_size: int = 5,
    debug_probs: bool = False,
) -> pd.DataFrame:
    """
    Long-form table for entropy change vs q and alpha, computed over common clonotypes.

    Returns columns:
      sample, percentile, Ts_ly49c, alpha, mean_entropy_change, n_common_clonotypes_ge5
    """
    base_summ = {}
    vc = df_s[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df_s[df_s[ct_col].isin(keep)]

    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        P = row_normalise(X, debug=debug_probs, tag=f"P_all[{ct}]")
        base_summ[ct] = {a: float(np.mean(renyi_entropy(P, alpha=float(a), axis=1))) for a in orders}

    rows = []
    for q in thresholds:
        df_f, Ts = apply_ly49c_filter_one_sample(df_s, ly49c_col, float(q))

        vc_f = df_f[ct_col].value_counts()
        keep_f = vc_f[vc_f >= min_size].index
        df_f2 = df_f[df_f[ct_col].isin(keep_f)]

        filt_summ = {}
        for ct, g in df_f2.groupby(ct_col):
            X = g[markers].to_numpy(dtype=float, copy=False)
            P = row_normalise(X, debug=debug_probs, tag=f"P_all[{ct}]")
            filt_summ[ct] = {a: float(np.mean(renyi_entropy(P, alpha=float(a), axis=1))) for a in orders}

        common = sorted(set(base_summ) & set(filt_summ))
        for a in orders:
            if common:
                dH = [filt_summ[ct][a] - base_summ[ct][a] for ct in common]
                mean_dH = float(np.mean(dH))
                n_common = int(len(common))
            else:
                mean_dH = np.nan
                n_common = 0

            rows.append({
                "sample": sample_name,
                "percentile": float(q),
                "Ts_ly49c": float(Ts),
                "alpha": float(a),
                "mean_entropy_change": mean_dH,
                "n_common_clonotypes_ge5": n_common,
                "n_cells_after": int(len(df_f)),
            })

    return pd.DataFrame(rows)

In [85]:
def clonotype_raw_and_prop_from_raw(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
) -> tuple[dict, dict, dict]:
    """
    For clonotypes with >=min_size cells:
      raw_mean[ct] = mean of raw peptide counts per cell
      prop[ct]     = raw_mean / raw_mean.sum() (i.e., normalized pattern)
      n_cells[ct]  = number of cells in clonotype (in THIS df)
    """
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]

    raw_mean, prop = {}, {}
    n_cells = {k: int(v) for k, v in vc[keep].to_dict().items()}

    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        m = X.mean(axis=0)
        raw_mean[ct] = m
        tot = m.sum()
        prop[ct] = m / tot if tot > 0 else np.zeros_like(m)

    return raw_mean, prop, n_cells

In [86]:
def plot_raw_and_prop_before_after_4x8_pdf(
    raw_b: dict, raw_a: dict,
    prop_b: dict, prop_a: dict,
    n_b: dict, n_a: dict,
    peptides: list[str],
    out_pdf: Path,
    title: str,
):
    xs = np.arange(len(peptides))
    w = 0.42
    per_page = 16

    clonotypes = sorted(n_b.keys(), key=lambda c: n_b[c], reverse=True)

    with PdfPages(out_pdf) as pdf:
        for start in range(0, len(clonotypes), per_page):
            chunk = clonotypes[start:start + per_page]

            fig, axes = plt.subplots(4, 8, figsize=(24, 12))
            axes = axes.flatten()

            for i, ct in enumerate(chunk):
                ax_raw = axes[2*i]
                ax_prp = axes[2*i + 1]
                has_after = ct in raw_a

                # Mean raw counts per cell
                b = raw_b[ct]
                if has_after:
                    a = raw_a[ct]
                    ax_raw.bar(xs - w/2, b, w, alpha=0.85, label="Before")
                    ax_raw.bar(xs + w/2, a, w, alpha=0.85, label="After")
                    ax_raw.set_title(f"{ct}\nMEAN n:{n_b[ct]}→{n_a.get(ct,0)}", fontsize=7)
                else:
                    ax_raw.bar(xs, b, w*1.8, alpha=0.6, label="Before (lost)")
                    ax_raw.set_title(f"{ct}\nMEAN n:{n_b[ct]}→0 LOST", fontsize=7, color="darkred")

                ax_raw.set_xticks(xs)
                ax_raw.set_xticklabels(peptides, rotation=90, fontsize=6)
                ax_raw.tick_params(axis="y", labelsize=6)
                ax_raw.spines["top"].set_visible(False)
                ax_raw.spines["right"].set_visible(False)
                if i == 0:
                    ax_raw.legend(fontsize=7, loc="upper right")

                # Proportions
                b = prop_b[ct]
                if has_after:
                    a = prop_a[ct]
                    ax_prp.bar(xs - w/2, b, w, alpha=0.85, label="Before")
                    ax_prp.bar(xs + w/2, a, w, alpha=0.85, label="After")
                    ax_prp.set_title(f"{ct}\nPROP n:{n_b[ct]}→{n_a.get(ct,0)}", fontsize=7)
                else:
                    ax_prp.bar(xs, b, w*1.8, alpha=0.6, label="Before (lost)")
                    ax_prp.set_title(f"{ct}\nPROP n:{n_b[ct]}→0 LOST", fontsize=7, color="darkred")

                ax_prp.set_xticks(xs)
                ax_prp.set_xticklabels(peptides, rotation=90, fontsize=6)
                ax_prp.tick_params(axis="y", labelsize=6)
                ax_prp.spines["top"].set_visible(False)
                ax_prp.spines["right"].set_visible(False)

            for j in range(2 * len(chunk), 32):
                axes[j].axis("off")

            page = start // per_page + 1
            n_pages = (len(clonotypes) + per_page - 1) // per_page
            fig.suptitle(f"{title} (page {page}/{n_pages})", fontsize=14, y=0.995)
            fig.tight_layout(rect=[0, 0, 1, 0.97])
            pdf.savefig(fig)
            plt.close(fig)

In [87]:
def renyi_profile_all_clonotypes_ge5(
    df_s: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
    orders: list[float] | None = None,
    debug: bool = False,
) -> pd.DataFrame:
    """
    For ALL clonotypes with >= min_size cells in ONE sample:
      compute mean Renyi entropy per clonotype for each alpha in `orders`.

    Returns long-form: clonotype, n_cells, alpha, renyi_entropy_mean
    """
    if orders is None:
        orders = [0, 1, 2, 3, 4, 5]

    vc = df_s[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df_s[df_s[ct_col].isin(keep)]

    rows = []
    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        P = row_normalise(X, debug=debug, tag=f"P_all[{ct}]")
        n_cells = int(P.shape[0])

        for a in orders:
            h = float(np.mean(renyi_entropy(P, alpha=float(a), axis=1)))
            rows.append({"clonotype": ct, "n_cells": n_cells, "alpha": float(a), "renyi_entropy_mean": h})

    return pd.DataFrame(rows)

In [88]:
def plot_renyi_spaghetti_all_clonotypes(
    df_long: pd.DataFrame,
    out_png: Path,
    title: str,
    alpha_linewidth: float = 1.7,
    alpha_opacity: float = 0.55,
):
    """
    df_long columns: clonotype, alpha, renyi_entropy_mean
    Plots one line per clonotype (spaghetti), darker & thicker.
    """
    fig, ax = plt.subplots(figsize=(8.2, 6.2))

    if df_long.empty or "clonotype" not in df_long.columns:
        ax.text(0.5, 0.5, "No clonotypes ≥ min_size", 
                ha="center", va="center", transform=ax.transAxes, fontsize=12)
        ax.set_xlabel("Renyi order α")
        ax.set_ylabel("Mean Renyi entropy (bits)")
        ax.set_title(title)
        fig.tight_layout()
        fig.savefig(out_png, dpi=170, bbox_inches="tight")
        plt.close(fig)
        return

    for ct, g in df_long.groupby("clonotype"):
        g2 = g.sort_values("alpha")
        ax.plot(
            g2["alpha"], g2["renyi_entropy_mean"],
            marker="o",
            lw=alpha_linewidth,
            alpha=alpha_opacity
        )

    ax.set_xlabel("Renyi order α")
    ax.set_ylabel("Mean Renyi entropy (bits)")
    ax.set_title(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [89]:
def plot_entropy_sweep_overlay_alpha_by_sample_2x3(
    df_long: pd.DataFrame,
    out_png: Path,
    title: str,
    chosen_q: dict[str, float] | None = None,
    max_panels: int = 6,
):
    """
    Expects df_long columns: sample, percentile, alpha, mean_entropy_change
    Produces 2x3 grid of samples; within each panel overlays α curves.
    If chosen_q is a dict, draws vertical line at sample-specific threshold.
    """
    samples = list(df_long["sample"].dropna().unique())[:max_panels]

    fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=True)
    axes = axes.flatten()

    for i in range(6):
        ax = axes[i]
        if i >= len(samples):
            ax.axis("off")
            continue

        s = samples[i]
        sub = df_long[df_long["sample"] == s].copy()
        for a, g in sub.groupby("alpha"):
            g2 = g.sort_values("percentile")
            ax.plot(g2["percentile"], g2["mean_entropy_change"], marker="o", lw=1.6, alpha=0.9, label=f"α={a:g}")

        if chosen_q is not None and s in chosen_q:
            ax.axvline(chosen_q[s], color="red", ls="--", alpha=0.7)

        ax.set_title(str(s))
        ax.set_xlabel("q (Ly49C percentile)")
        ax.invert_xaxis()
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(frameon=False, fontsize=8, ncol=2)

    fig.suptitle(title, y=0.98)
    fig.text(0.5, 0.04, "q (Ly49C percentile, within sample)", ha="center")
    fig.text(0.04, 0.5, "Mean ΔHα (after − before) [bits]", va="center", rotation="vertical")
    fig.tight_layout(rect=[0.06, 0.06, 1, 0.95])
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [90]:
def coherence_before_after_table(
    df_before: pd.DataFrame,
    df_after: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
    debug_probs: bool = False,
) -> pd.DataFrame:
    base = compute_clonotype_summaries(
        df_before, ct_col, markers, min_size=min_size, entropy_alpha=1.0, debug_probs=debug_probs
    )
    aft = compute_clonotype_summaries(
        df_after, ct_col, markers, min_size=min_size, entropy_alpha=1.0, debug_probs=debug_probs
    )

    common = sorted(set(base) & set(aft))
    
    if not common:
        return pd.DataFrame(columns=["clonotype", "coh_before", "coh_after", "n_before", "n_after"])
    
    rows = []
    for ct in common:
        rows.append({
            "clonotype": ct,
            "coh_before": float(base[ct]["mean_coherence"]),
            "coh_after": float(aft[ct]["mean_coherence"]),
            "n_before": int(base[ct]["n_cells"]),
            "n_after": int(aft[ct]["n_cells"]),
        })
    return pd.DataFrame(rows)

In [91]:
def plot_coherence_before_after_scatter(
    df_pairs: pd.DataFrame,
    out_png: Path,
    title: str,
    mean_coherence_change: float | None = None,
):
    """
    Scatter plot of coherence before vs after filtering.
    Includes mean coherence change annotation if provided.
    """
    fig, ax = plt.subplots(figsize=(6.5, 6.5))

    if df_pairs.empty or "coh_before" not in df_pairs.columns:
        ax.text(0.5, 0.5, "No common clonotypes ≥ min_size", 
                ha="center", va="center", transform=ax.transAxes, fontsize=12)
        ax.set_xlabel("Mean coherence (baseline)")
        ax.set_ylabel("Mean coherence (filtered)")
        ax.set_title(title)
        fig.tight_layout()
        fig.savefig(out_png, dpi=170, bbox_inches="tight")
        plt.close(fig)
        return

    x = df_pairs["coh_before"].to_numpy()
    y = df_pairs["coh_after"].to_numpy()

    ax.scatter(x, y, s=18, alpha=0.7)

    lo = np.nanmin(np.r_[x, y])
    hi = np.nanmax(np.r_[x, y])
    ax.plot([lo, hi], [lo, hi], ls="--", color="black", alpha=0.6)

    ax.set_xlabel("Mean coherence (baseline)")
    ax.set_ylabel("Mean coherence (filtered)")
    ax.set_title(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    annot_lines = []
    if len(df_pairs) >= 3 and np.isfinite(x).all() and np.isfinite(y).all():
        r = float(np.corrcoef(x, y)[0, 1])
        annot_lines.append(f"r = {r:.2f}")
    annot_lines.append(f"N = {len(df_pairs)}")
    
    if mean_coherence_change is not None and np.isfinite(mean_coherence_change):
        sign = "+" if mean_coherence_change >= 0 else ""
        annot_lines.append(f"Mean Δcoh = {sign}{mean_coherence_change:.3f}")
    
    ax.text(0.02, 0.98, "\n".join(annot_lines), transform=ax.transAxes,
            va="top", ha="left", fontsize=10)

    fig.tight_layout()
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [92]:
def plot_combined_coherence(
    before_dfs: dict[str, pd.DataFrame],
    after_dfs: dict[str, pd.DataFrame],
    samples_to_combine: list[str],
    markers: list[str],
    output_path: Path,
    title: str,
    ct_col: str = CT_COL,
    min_size: int = 5,
    debug_probs: bool = False,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Combine pre-filtered dataframes and compute coherence before vs after.
    
    Parameters
    ----------
    before_dfs : dict[str, pd.DataFrame]
        Sample -> dataframe before Ly49C filtering
    after_dfs : dict[str, pd.DataFrame]
        Sample -> dataframe after Ly49C filtering
    samples_to_combine : list[str]
        List of sample names to combine
    markers : list[str]
        Dextramer marker columns
    output_path : Path
        Full path for output files (without extension)
    title : str
        Plot title
    
    Returns
    -------
    pd.DataFrame
        Coherence pairs table
    """
    df_before = pd.concat([before_dfs[s] for s in samples_to_combine], ignore_index=True)
    df_after = pd.concat([after_dfs[s] for s in samples_to_combine], ignore_index=True)
    
    if verbose:
        print(f"  Combined: {len(df_before):,} -> {len(df_after):,} cells")
    
    pairs = coherence_before_after_table(
        df_before=df_before,
        df_after=df_after,
        ct_col=ct_col,
        markers=markers,
        min_size=min_size,
        debug_probs=debug_probs,
    )
    
    if not pairs.empty:
        mean_coh_change = float((pairs["coh_after"] - pairs["coh_before"]).mean())
    else:
        mean_coh_change = None
    
    output_path = Path(output_path)
    pairs.to_csv(output_path.with_suffix(".csv"), index=False)
    
    plot_coherence_before_after_scatter(
        pairs,
        out_png=output_path.with_suffix(".png"),
        title=title,
        mean_coherence_change=mean_coh_change,
    )
    
    if verbose:
        print(f"  Saved: {output_path.with_suffix('.png')}")
        if mean_coh_change is not None:
            print(f"  Mean Δcoherence: {mean_coh_change:+.4f}")
        print(f"  N clonotypes ≥{min_size}: {len(pairs)}")
    
    return pairs

In [93]:
def plot_all_clonotypes_by_coherence_pdf(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    out_pdf: Path,
    title_prefix: str = "",
    min_size: int = 5,
    normalize: bool = True,
    jitter_width: float = 0.25,
    plots_per_page: int = 12,
    debug_probs: bool = False,
):
    """
    Multi-page PDF with one panel per clonotype, ordered by decreasing coherence.
    
    Parameters
    ----------
    df : pd.DataFrame
        Data containing cells
    ct_col : str
        Column name for clonotype
    markers : list[str]
        Dextramer marker column names
    peptides : list[str]
        Short peptide names for x-axis labels
    out_pdf : Path
        Output PDF path
    title_prefix : str
        Prefix for the PDF suptitle (e.g., dataset/sample name)
    min_size : int
        Minimum cells per clonotype to include
    normalize : bool
        If True, plot proportions. If False, plot raw counts.
    plots_per_page : int
        Number of clonotype panels per page (e.g., 12 = 3x4 grid)
    """
    out_pdf = Path(out_pdf)
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    
    # Compute coherence for all clonotypes >= min_size
    summaries = compute_clonotype_summaries(
        df, ct_col, markers, min_size=min_size, entropy_alpha=1.0, debug_probs=debug_probs
    )
    
    if not summaries:
        print("No clonotypes >= min_size")
        return
    
    # Build list sorted by decreasing coherence
    coh_list = [
        (ct, s["mean_coherence"], s["n_cells"]) 
        for ct, s in summaries.items() 
        if np.isfinite(s["mean_coherence"])
    ]
    coh_list.sort(key=lambda x: x[1], reverse=True)
    
    n_clonotypes = len(coh_list)
    print(f"Plotting {n_clonotypes} clonotypes ordered by decreasing coherence")
    
    if plots_per_page == 12:
        nrows, ncols = 3, 4
    elif plots_per_page == 16:
        nrows, ncols = 4, 4
    elif plots_per_page == 9:
        nrows, ncols = 3, 3
    else:
        ncols = 4
        nrows = (plots_per_page + ncols - 1) // ncols
    
    n_pages = (n_clonotypes + plots_per_page - 1) // plots_per_page
    
    ylabel = "Proportion" if normalize else "Raw count"
    xs = np.arange(len(peptides))
    
    with PdfPages(out_pdf) as pdf:
        for page_idx in range(n_pages):
            start = page_idx * plots_per_page
            end = min(start + plots_per_page, n_clonotypes)
            chunk = coh_list[start:end]
            
            fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
            axes = axes.flatten()
            
            for i, (ct, coh, n_cells) in enumerate(chunk):
                ax = axes[i]
                
                df_ct = df[df[ct_col] == ct]
                X = df_ct[markers].to_numpy(dtype=float)
                
                if normalize:
                    row_sums = X.sum(axis=1, keepdims=True)
                    row_sums[row_sums == 0] = 1.0
                    X = X / row_sums
                
                for j in range(X.shape[0]):
                    jitter = np.random.uniform(-jitter_width, jitter_width, len(peptides))
                    ax.scatter(xs + jitter, X[j, :], alpha=0.5, s=15, edgecolors="none")
                
                mean_vals = X.mean(axis=0)
                ax.scatter(xs, mean_vals, marker="D", s=40, color="black", zorder=10)
                
                ax.set_xticks(xs)
                ax.set_xticklabels(peptides, rotation=90, fontsize=6)
                ax.tick_params(axis="y", labelsize=7)
                ax.set_ylabel(ylabel, fontsize=7)
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)
                
                rank = start + i + 1
                ax.set_title(f"#{rank} {ct}\ncoh={coh:.3f}, n={n_cells}", fontsize=8)
            
            for j in range(len(chunk), len(axes)):
                axes[j].axis("off")
            
            fig.suptitle(
                f"{title_prefix} | Clonotypes by coherence (page {page_idx + 1}/{n_pages})",
                fontsize=12, y=0.995
            )
            fig.tight_layout(rect=[0, 0, 1, 0.97])
            pdf.savefig(fig)
            plt.close(fig)
    
    print(f"Saved: {out_pdf}")

In [94]:
def plot_sweep_metrics_2x3(
    sweep_df: pd.DataFrame,
    sweep_entropy_df: pd.DataFrame,
    out_png: Path,
    title: str,
    sample_q: float | None = None,
):
    """
    2x3 panel plot showing sweep metrics:
      [0,0] Median cosine distance
      [0,1] Median L1 distance
      [0,2] Mean entropy change (α=0,1,2,3,4,5 overlaid)
      [1,0] Mean coherence change
      [1,1] Data retention (% cells retained)
      [1,2] Number of common clonotypes ≥ min_size
    """
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
    
    q_vals = sweep_df["percentile"].to_numpy()
    
    # [0,0] Median cosine distance
    ax = axes[0, 0]
    ax.plot(q_vals, sweep_df["median_cosine_dist"], marker="o", lw=1.8, color="C0")
    ax.set_ylabel("Median cosine distance")
    ax.set_title("Pattern shift (cosine)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if sample_q is not None:
        ax.axvline(sample_q, color="red", ls="--", alpha=0.7)
    ax.invert_xaxis()
    
    # [0,1] Median L1 distance
    ax = axes[0, 1]
    ax.plot(q_vals, sweep_df["median_l1_dist"], marker="o", lw=1.8, color="C1")
    ax.set_ylabel("Median L1 distance")
    ax.set_title("Pattern shift (L1)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if sample_q is not None:
        ax.axvline(sample_q, color="red", ls="--", alpha=0.7)
    
    # [0,2] Mean entropy change (α=0..5 overlaid)
    ax = axes[0, 2]
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, 6))
    for i, alpha_val in enumerate([0, 1, 2, 3, 4, 5]):
        sub = sweep_entropy_df[sweep_entropy_df["alpha"] == alpha_val].sort_values("percentile")
        if not sub.empty:
            ax.plot(sub["percentile"], sub["mean_entropy_change"], 
                   marker="o", lw=1.5, alpha=0.85, color=colors[i], label=f"α={alpha_val}")
    ax.set_ylabel("Mean ΔH (bits)")
    ax.set_title("Entropy change by α")
    ax.legend(frameon=False, fontsize=8, ncol=2, loc="best")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if sample_q is not None:
        ax.axvline(sample_q, color="red", ls="--", alpha=0.7)
    
    # [1,0] Mean coherence change
    ax = axes[1, 0]
    ax.plot(q_vals, sweep_df["mean_coherence_change"], marker="o", lw=1.8, color="C2")
    ax.axhline(0, color="gray", ls=":", alpha=0.5)
    ax.set_ylabel("Mean Δcoherence")
    ax.set_title("Coherence change")
    ax.set_xlabel("q (Ly49C percentile)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if sample_q is not None:
        ax.axvline(sample_q, color="red", ls="--", alpha=0.7)
    
    # [1,1] Data retention
    ax = axes[1, 1]
    ax.plot(q_vals, sweep_df["pct_cells_retained"], marker="o", lw=1.8, color="C3")
    ax.set_ylabel("% cells retained")
    ax.set_title("Data retention")
    ax.set_xlabel("q (Ly49C percentile)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if sample_q is not None:
        ax.axvline(sample_q, color="red", ls="--", alpha=0.7)
    
    # [1,2] Number of common clonotypes
    ax = axes[1, 2]
    ax.plot(q_vals, sweep_df["n_common_clonotypes_ge5"], marker="o", lw=1.8, color="C4")
    ax.set_ylabel("N common clonotypes ≥5")
    ax.set_title("Clonotype retention")
    ax.set_xlabel("q (Ly49C percentile)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if sample_q is not None:
        ax.axvline(sample_q, color="red", ls="--", alpha=0.7)
    
    fig.suptitle(title, fontsize=12, y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [95]:
def run_samplewise_pipeline(
    df: pd.DataFrame,
    dataset_name: str,
    markers: list[str],
    peptides: list[str],
    output_root: Path,
    thresholds: np.ndarray,
    chosen_q: dict[str, float],
    ct_col: str = CT_COL,
    sample_col: str = SAMPLE_COL,
    ly49c_col: str = LY49C_COL,
    min_size: int = 5,
    entropy_alpha_for_sweep: float = 1.0,
    renyi_orders: list[float] | None = None,
    debug_probs: bool = False,
    filter_zero_dextramer: bool = True,
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:
    """
    Parameters
    ----------
    chosen_q : dict[str, float]
        Maps sample name -> Ly49C percentile threshold.
        All samples in the data must have an entry (raises KeyError otherwise).
    
    Returns
    -------
    index_df : pd.DataFrame
        Index of all outputs
    before_dfs : dict[str, pd.DataFrame]
        Sample name -> dataframe before Ly49C filtering (after zero-dex removal)
    after_dfs : dict[str, pd.DataFrame]
        Sample name -> dataframe after Ly49C filtering
    """
    output_root = Path(output_root)
    ds_out = output_root / dataset_name
    ds_out.mkdir(parents=True, exist_ok=True)

    orders = renyi_orders or [0, 1, 2, 3, 4, 5]

    needed = {ct_col, sample_col, ly49c_col, *markers}
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns (first few): {missing[:10]}")

    if filter_zero_dextramer:
        if verbose:
            print(f"Dataset: {dataset_name}")
        df = filter_zero_dextramer_cells(df, markers, verbose=verbose)

    index_rows = []
    sweep_entropy_all_samples = []
    before_dfs = {}
    after_dfs = {} 

    for sample, df_s in df.groupby(sample_col):
        s_out = ds_out / str(sample)
        s_out.mkdir(parents=True, exist_ok=True)

        sample_q = chosen_q[sample]

        if verbose:
            print(f"  Processing sample: {sample} ({len(df_s):,} cells, q={sample_q})")

        before_dfs[sample] = df_s.copy()

        sweep_df = ly49c_sweep_metrics_one_sample(
            df_s=df_s,
            ct_col=ct_col,
            ly49c_col=ly49c_col,
            markers=markers,
            thresholds=thresholds,
            min_size=min_size,
            entropy_alpha=entropy_alpha_for_sweep,
            debug_probs=debug_probs,
        )
        sweep_csv = s_out / f"ly49c_sweep_metrics_alpha{entropy_alpha_for_sweep}.csv"
        sweep_df.to_csv(sweep_csv, index=False)

        sweep_ent_long = ly49c_sweep_entropy_orders_one_sample(
            df_s=df_s,
            sample_name=str(sample),
            ct_col=ct_col,
            ly49c_col=ly49c_col,
            markers=markers,
            thresholds=thresholds,
            orders=orders,
            min_size=min_size,
            debug_probs=debug_probs,
        )
        sweep_entropy_all_samples.append(sweep_ent_long)

        sweep_metrics_png = s_out / "ly49c_sweep_metrics_2x3.png"
        sweep_ent_this_sample = sweep_ent_long[sweep_ent_long["sample"] == str(sample)]
        plot_sweep_metrics_2x3(
            sweep_df=sweep_df,
            sweep_entropy_df=sweep_ent_this_sample,
            out_png=sweep_metrics_png,
            title=f"{dataset_name} | {sample} | Ly49C sweep metrics",
            sample_q=sample_q,
        )

        df_after, Ts = apply_ly49c_filter_one_sample(df_s, ly49c_col, sample_q)

        after_dfs[sample] = df_after.copy()

        raw_b, prop_b, n_b = clonotype_raw_and_prop_from_raw(df_s, ct_col, markers, min_size=min_size)
        raw_a, prop_a, n_a = clonotype_raw_and_prop_from_raw(df_after, ct_col, markers, min_size=min_size)

        pdf_out = s_out / f"clonotypes_MEAN_and_PROP_before_after_q{sample_q:.3f}.pdf"
        plot_raw_and_prop_before_after_4x8_pdf(
            raw_b, raw_a, prop_b, prop_a, n_b, n_a,
            peptides=peptides,
            out_pdf=pdf_out,
            title=f"{dataset_name} | {sample} | clonotypes≥{min_size} MEAN+PROP before/after (q={sample_q:.3f}, Ts={Ts:.2f})",
        )

        renyi_base = renyi_profile_all_clonotypes_ge5(
            df_s=df_s, ct_col=ct_col, markers=markers, min_size=min_size, orders=orders, debug=debug_probs
        )
        renyi_after = renyi_profile_all_clonotypes_ge5(
            df_s=df_after, ct_col=ct_col, markers=markers, min_size=min_size, orders=orders, debug=debug_probs
        )

        renyi_base_csv = s_out / f"renyi_all_ge{min_size}_baseline.csv"
        renyi_after_csv = s_out / f"renyi_all_ge{min_size}_filtered_q{sample_q:.3f}.csv"
        renyi_base.to_csv(renyi_base_csv, index=False)
        renyi_after.to_csv(renyi_after_csv, index=False)

        plot_renyi_spaghetti_all_clonotypes(
            renyi_base,
            out_png=s_out / "renyi_spaghetti_baseline.png",
            title=f"{dataset_name} | {sample} | Renyi spaghetti (ALL clonotypes≥{min_size}) | baseline",
            alpha_linewidth=1.8,
            alpha_opacity=0.6,
        )
        plot_renyi_spaghetti_all_clonotypes(
            renyi_after,
            out_png=s_out / f"renyi_spaghetti_filtered_q{sample_q:.3f}.png",
            title=f"{dataset_name} | {sample} | Renyi spaghetti (ALL clonotypes≥{min_size}) | filtered q={sample_q:.3f}",
            alpha_linewidth=1.8,
            alpha_opacity=0.6,
        )

        pairs = coherence_before_after_table(
            df_before=df_s,
            df_after=df_after,
            ct_col=ct_col,
            markers=markers,
            min_size=min_size,
            debug_probs=debug_probs,
        )
        coh_csv = s_out / f"coherence_before_vs_after_q{sample_q:.3f}.csv"
        coh_png = s_out / f"coherence_before_vs_after_q{sample_q:.3f}.png"
        pairs.to_csv(coh_csv, index=False)

        chosen_row = sweep_df[np.isclose(sweep_df["percentile"], sample_q)]
        if not chosen_row.empty:
            mean_coh_change = float(chosen_row["mean_coherence_change"].iloc[0])
        else:
            mean_coh_change = None

        plot_coherence_before_after_scatter(
            pairs,
            out_png=coh_png,
            title=f"{dataset_name} | {sample} | coherence before vs after (q={sample_q:.3f})",
            mean_coherence_change=mean_coh_change,
        )

        coh_by_clonotype_pdf = s_out / f"clonotypes_by_coherence_q{sample_q:.3f}.pdf"
        plot_all_clonotypes_by_coherence_pdf(
            df=df_after,
            ct_col=ct_col,
            markers=markers,
            peptides=peptides,
            out_pdf=coh_by_clonotype_pdf,
            title_prefix=f"{dataset_name} | {sample} | q={sample_q:.3f}",
            min_size=min_size,
            normalize=True,
            plots_per_page=12,
            debug_probs=debug_probs,
        )

        index_rows.append({
            "dataset": dataset_name,
            "sample": sample,
            "n_cells_before": int(len(df_s)),
            "n_cells_after": int(len(df_after)),
            "n_clonotypes_ge5_before": int(len(n_b)),
            "n_clonotypes_ge5_after": int(len(n_a)),
            "chosen_q": float(sample_q),
            "Ts": float(Ts),
            "sweep_csv": str(sweep_csv),
            "sweep_metrics_png": str(sweep_metrics_png),
            "before_after_pdf": str(pdf_out),
            "renyi_base_csv": str(renyi_base_csv),
            "renyi_after_csv": str(renyi_after_csv),
            "coherence_csv": str(coh_csv),
            "coherence_png": str(coh_png),
            "clonotypes_by_coherence_pdf": str(coh_by_clonotype_pdf)
        })

    if sweep_entropy_all_samples:
        sweep_all = pd.concat(sweep_entropy_all_samples, ignore_index=True)
        plot_entropy_sweep_overlay_alpha_by_sample_2x3(
            sweep_all,
            out_png=ds_out / "ly49c_entropy_sweep_overlay_alpha0to5_2x3.png",
            title=f"{dataset_name} | ΔHα(q) sweep overlay α=0..5 (first 6 samples)",
            chosen_q=chosen_q,
        )

    index_df = pd.DataFrame(index_rows)
    index_df.to_csv(ds_out / "INDEX.csv", index=False)
    
    return index_df, before_dfs, after_dfs

In [96]:
df_b10br = pd.read_csv(
    "../Data/20260116 Comparison 3/20251223 BL6-B10BR HTx HIL Clonotypes with ADT Counts.csv",
    index_col=0
)

df_balbc = pd.read_csv(
    "../Data/20260116 Comparison 3/20251218 BL6-BALBc HTx HIL Clonotypes with ADT Counts.csv",
    index_col=0
)

df_d7abc = pd.read_csv(
    "../Data/20260116 Comparison 3/20260114 B10BR D7ABC Kb LL Repertoire + ADT counts.csv"
)

In [97]:
chosen_q_b10br = {
    "BL6-B10BR_HTxA": 0.975,
    "BL6-B10BR_HTxB": 0.975,
    "BL6-B10BR_HTxC": 0.925,
}

chosen_q_balbc = {
    "BL6-BALBc_HTxA": 1.00,
    "BL6-BALBc_HTxB": 0.975,
    "BL6-BALBc_HTxC": 0.95,
}

index_b10br, before_b10br, after_b10br = run_samplewise_pipeline(
    df=df_b10br,
    dataset_name="B10BR_HIL",
    markers=markers_b10br,
    peptides=peptides_b10br,
    output_root=Path("Comparison3_Samplewise_Outputs"),
    thresholds=thresholds,
    chosen_q=chosen_q_b10br,
    min_size=5,
    entropy_alpha_for_sweep=1.0,
    renyi_orders=[0, 1, 2, 3, 4, 5],
    debug_probs=False,
)

index_balbc, before_balbc, after_balbc = run_samplewise_pipeline(
    df=df_balbc,
    dataset_name="BALBc_HIL",
    markers=markers_balbc,
    peptides=peptides_balbc,
    output_root=Path("Comparison3_Samplewise_Outputs"),
    thresholds=thresholds,
    chosen_q=chosen_q_balbc,
    min_size=5,
    entropy_alpha_for_sweep=1.0,
    renyi_orders=[0, 1, 2, 3, 4, 5],
    debug_probs=False,
)

pairs_b10br_combined = plot_combined_coherence(
    before_dfs=before_b10br,
    after_dfs=after_b10br,
    samples_to_combine=["BL6-B10BR_HTxA", "BL6-B10BR_HTxB", "BL6-B10BR_HTxC"],
    markers=markers_b10br,
    output_path=Path("Comparison3_Samplewise_Outputs/B10BR_HIL/coherence_combined_all"),
    title="B10BR_HIL | Combined (HTxA + HTxB + HTxC)",
    min_size=5,
)

pairs_balbc_combined = plot_combined_coherence(
    before_dfs=before_balbc,
    after_dfs=after_balbc,
    samples_to_combine=["BL6-BALBc_HTxB", "BL6-BALBc_HTxC"],
    markers=markers_balbc,
    output_path=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/coherence_combined_HTxB_HTxC"),
    title="BALBc_HIL | Combined (HTxB + HTxC)",
    min_size=5,
)

Dataset: B10BR_HIL
[filter_zero_dextramer] Removed 110 / 18,924 cells (0.58%) with zero total dextramer counts
  Processing sample: BL6-B10BR_HTxA (541 cells, q=0.975)
Plotting 18 clonotypes ordered by decreasing coherence
Saved: Comparison3_Samplewise_Outputs/B10BR_HIL/BL6-B10BR_HTxA/clonotypes_by_coherence_q0.975.pdf
  Processing sample: BL6-B10BR_HTxB (7,458 cells, q=0.975)
Plotting 210 clonotypes ordered by decreasing coherence
Saved: Comparison3_Samplewise_Outputs/B10BR_HIL/BL6-B10BR_HTxB/clonotypes_by_coherence_q0.975.pdf
  Processing sample: BL6-B10BR_HTxC (10,815 cells, q=0.925)
Plotting 238 clonotypes ordered by decreasing coherence
Saved: Comparison3_Samplewise_Outputs/B10BR_HIL/BL6-B10BR_HTxC/clonotypes_by_coherence_q0.925.pdf
Dataset: BALBc_HIL
[filter_zero_dextramer] Removed 112 / 9,315 cells (1.20%) with zero total dextramer counts
  Processing sample: BL6-BALBc_HTxA (2,696 cells, q=1.0)
Plotting 111 clonotypes ordered by decreasing coherence
Saved: Comparison3_Samplewise

In [109]:
chosen_q_d7abc = {
    "D7A": 0.95,
    "D7B": 0.95,
    "D7C": 0.95,
}

index_d7abc, before_d7abc, after_d7abc = run_samplewise_pipeline(
    df=df_d7abc,
    dataset_name="D7ABC_LL_LY49C_FILTER",
    markers=markers_b10br,
    peptides=peptides_b10br,
    output_root=Path("Comparison3_Samplewise_Outputs"),
    thresholds=thresholds,
    chosen_q=chosen_q_d7abc,
    min_size=5,
    entropy_alpha_for_sweep=1.0,
    renyi_orders=[0, 1, 2, 3, 4, 5],
    debug_probs=False,
)

index_d7abc, before_d7abc, after_d7abc = run_samplewise_pipeline(
    df=df_d7abc,
    dataset_name="D7ABC_LL_EEPVKKI_FILTER",
    markers=markers_b10br,
    peptides=peptides_b10br,
    output_root=Path("Comparison3_Samplewise_Outputs"),
    thresholds=thresholds,
    chosen_q=chosen_q_d7abc,
    ly49c_col="RiO-Allo:H-2Kb-EEEPVKKI-pAbO",
    min_size=5,
    entropy_alpha_for_sweep=1.0,
    renyi_orders=[0, 1, 2, 3, 4, 5],
    debug_probs=False,
)

Dataset: D7ABC_LL_LY49C_FILTER
[filter_zero_dextramer] Removed 2,739 / 17,991 cells (15.22%) with zero total dextramer counts
  Processing sample: D7A (5,213 cells, q=0.95)
Plotting 147 clonotypes ordered by decreasing coherence
Saved: Comparison3_Samplewise_Outputs/D7ABC_LL_LY49C_FILTER/D7A/clonotypes_by_coherence_q0.950.pdf
  Processing sample: D7B (4,787 cells, q=0.95)
Plotting 162 clonotypes ordered by decreasing coherence
Saved: Comparison3_Samplewise_Outputs/D7ABC_LL_LY49C_FILTER/D7B/clonotypes_by_coherence_q0.950.pdf
  Processing sample: D7C (5,252 cells, q=0.95)
Plotting 143 clonotypes ordered by decreasing coherence
Saved: Comparison3_Samplewise_Outputs/D7ABC_LL_LY49C_FILTER/D7C/clonotypes_by_coherence_q0.950.pdf
Dataset: D7ABC_LL_EEPVKKI_FILTER
[filter_zero_dextramer] Removed 2,739 / 17,991 cells (15.22%) with zero total dextramer counts
  Processing sample: D7A (5,213 cells, q=0.95)
Plotting 149 clonotypes ordered by decreasing coherence
Saved: Comparison3_Samplewise_Outputs

In [99]:
def classify_clonotype_specificity(
    prop_vector: np.ndarray,
    peptides: list[str],
    threshold: float = 0.90,
) -> dict:
    """
    Classify a clonotype as single, dual, or multiple specificity based on
    the proportion of binding concentrated in the top peptide(s).
    
    Parameters
    ----------
    prop_vector : np.ndarray
        Normalized binding proportions per peptide (sums to 1)
    peptides : list[str]
        Peptide names corresponding to prop_vector
    threshold : float
        Proportion threshold for classification (default 0.90)
    
    Returns
    -------
    dict with keys:
        - classification: "single", "dual", or "multiple"
        - top_peptides: list of dominant peptide(s)
        - top_proportions: list of their proportions
        - cumulative_proportion: sum of top peptide(s)
        - n_peptides_to_threshold: how many peptides needed to reach threshold
    """
    order = np.argsort(prop_vector)[::-1]
    sorted_props = prop_vector[order]
    sorted_peptides = [peptides[i] for i in order]
    
    # Single specificity?
    if sorted_props[0] >= threshold:
        return {
            "classification": "single",
            "top_peptides": [sorted_peptides[0]],
            "top_proportions": [float(sorted_props[0])],
            "cumulative_proportion": float(sorted_props[0]),
            "n_peptides_to_threshold": 1,
        }
    
    # Dual specificity?
    if sorted_props[0] + sorted_props[1] >= threshold:
        return {
            "classification": "dual",
            "top_peptides": [sorted_peptides[0], sorted_peptides[1]],
            "top_proportions": [float(sorted_props[0]), float(sorted_props[1])],
            "cumulative_proportion": float(sorted_props[0] + sorted_props[1]),
            "n_peptides_to_threshold": 2,
        }
    
    # Multiple specificity?
    cumsum = np.cumsum(sorted_props)
    n_needed = int(np.searchsorted(cumsum, threshold) + 1)
    n_needed = min(n_needed, len(sorted_props))
    
    return {
        "classification": "multiple",
        "top_peptides": sorted_peptides[:n_needed],
        "top_proportions": [float(p) for p in sorted_props[:n_needed]],
        "cumulative_proportion": float(cumsum[n_needed - 1]) if n_needed <= len(cumsum) else float(cumsum[-1]),
        "n_peptides_to_threshold": n_needed,
    }

In [100]:
def classify_all_clonotypes(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    min_size: int = 5,
    threshold: float = 0.90,
) -> pd.DataFrame:
    """
    Classify all clonotypes in a dataset by specificity.
    
    Parameters
    ----------
    df : pd.DataFrame
        Data with cells as rows
    ct_col : str
        Clonotype column name
    markers : list[str]
        Dextramer marker columns
    peptides : list[str]
        Short peptide names
    min_size : int
        Minimum cells per clonotype
    threshold : float
        Proportion threshold for classification
    
    Returns
    -------
    pd.DataFrame with columns:
        clonotype, n_cells, classification, top_peptides, top_proportions,
        cumulative_proportion, n_peptides_to_threshold, prop_vector
    """
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]
    
    rows = []
    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float)
        
        mean_counts = X.mean(axis=0)
        total = mean_counts.sum()
        prop_vector = mean_counts / total if total > 0 else np.zeros_like(mean_counts)
        
        result = classify_clonotype_specificity(prop_vector, peptides, threshold=threshold)
        
        rows.append({
            "clonotype": ct,
            "n_cells": int(len(g)),
            "classification": result["classification"],
            "top_peptides": result["top_peptides"],
            "top_proportions": result["top_proportions"],
            "cumulative_proportion": result["cumulative_proportion"],
            "n_peptides_to_threshold": result["n_peptides_to_threshold"],
            "prop_vector": prop_vector
        })
    
    return pd.DataFrame(rows)

In [101]:
def leave_one_out_coherence(
    P: np.ndarray,
    markers: list[str],
) -> dict[str, float]:
    """
    Compute coherence after removing each peptide/marker in turn.
    
    Parameters
    ----------
    P : np.ndarray
        Row-normalized binding matrix (n_cells x n_peptides)
    markers : list[str]
        Marker/peptide names
    
    Returns
    -------
    dict mapping marker -> coherence when that marker is excluded
    """
    n_cells, n_markers = P.shape
    
    if n_cells < 2:
        return {m: np.nan for m in markers}
    
    results = {}
    
    for i, marker in enumerate(markers):
        P_reduced = np.delete(P, i, axis=1)
        
        row_sums = P_reduced.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        P_renorm = P_reduced / row_sums
        
        coh = mean_pairwise_cosine_similarity(P_renorm)
        results[marker] = coh
    
    return results

In [102]:
def compute_clonotype_coherence_analysis(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    min_size: int = 5,
    min_prop_for_ranking: float = 0.05
) -> pd.DataFrame:
    """
    For each clonotype, compute:
      - Full coherence
      - Leave-one-out coherence for each peptide
      - Delta coherence (LOO - full) for each peptide
    
    Parameters
    ----------
    df : pd.DataFrame
        Data with cells as rows
    ct_col : str
        Clonotype column name
    markers : list[str]
        Dextramer marker columns
    peptides : list[str]
        Short peptide names (for output column names)
    min_size : int
        Minimum cells per clonotype
    min_prop_for_ranking : float
        Only consider peptides with >= this proportion when determining
        most_disruptive / most_stabilizing. Default 0.05 (5%).
    
    Returns
    -------
    pd.DataFrame with columns:
        clonotype, n_cells, coherence_full,
        coh_without_{peptide} for each peptide,
        delta_coh_{peptide} for each peptide,
        most_disruptive_peptide (removing it raises coherence most),
        most_stabilizing_peptide (removing it drops coherence most)
    """
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]
    
    rows = []
    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float)
        P = row_normalise(X)
        
        mean_counts = X.mean(axis=0)
        total = mean_counts.sum()
        prop_vector = mean_counts / total if total > 0 else np.zeros_like(mean_counts)
        
        coh_full = mean_pairwise_cosine_similarity(P)
        
        loo_results = leave_one_out_coherence(P, markers)
        
        row = {
            "clonotype": ct,
            "n_cells": int(len(g)),
            "coherence_full": coh_full,
        }
        
        deltas = {}
        for i, (marker, peptide) in enumerate(zip(markers, peptides)):
            coh_without = loo_results[marker]
            delta = coh_without - coh_full if np.isfinite(coh_without) and np.isfinite(coh_full) else np.nan
            
            row[f"coh_without_{peptide}"] = coh_without
            row[f"delta_coh_{peptide}"] = delta
            row[f"prop_{peptide}"] = prop_vector[i]
            
            if prop_vector[i] >= min_prop_for_ranking and np.isfinite(delta):
                deltas[peptide] = delta
        
        if deltas:
            row["most_disruptive_peptide"] = max(deltas, key=deltas.get)
            row["most_disruptive_delta"] = max(deltas.values())
            row["most_stabilizing_peptide"] = min(deltas, key=deltas.get)
            row["most_stabilizing_delta"] = min(deltas.values())
        else:
            row["most_disruptive_peptide"] = None
            row["most_disruptive_delta"] = np.nan
            row["most_stabilizing_peptide"] = None
            row["most_stabilizing_delta"] = np.nan
        
        rows.append(row)
    
    return pd.DataFrame(rows)

In [103]:
def full_specificity_analysis(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    min_size: int = 5,
    threshold: float = 0.90,
    min_prop_for_ranking: float = 0.05
) -> pd.DataFrame:
    """
    Combine specificity classification with coherence analysis.
    """
    spec_df = classify_all_clonotypes(
        df=df,
        ct_col=ct_col,
        markers=markers,
        peptides=peptides,
        min_size=min_size,
        threshold=threshold,
    )
    
    coh_df = compute_clonotype_coherence_analysis(
        df=df,
        ct_col=ct_col,
        markers=markers,
        peptides=peptides,
        min_size=min_size,
        min_prop_for_ranking=min_prop_for_ranking
    )
    
    merged = spec_df.merge(coh_df, on=["clonotype", "n_cells"], how="outer")
    
    return merged

In [104]:
df_b10br_combined = pd.concat([after_b10br[s] for s in ["BL6-B10BR_HTxA", "BL6-B10BR_HTxB", "BL6-B10BR_HTxC"]], ignore_index=True)
df_balbc_combined = pd.concat([after_balbc[s] for s in ["BL6-BALBc_HTxB", "BL6-BALBc_HTxC"]], ignore_index=True)


specificity_b10br = full_specificity_analysis(
    df=df_b10br_combined,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    min_size=5,
    threshold=0.85
)

specificity_balbc = full_specificity_analysis(
    df=df_balbc_combined,
    ct_col=CT_COL,
    markers=markers_balbc,
    peptides=peptides_balbc,
    min_size=5,
    threshold=0.85
)

out_path_b10br = Path(f"Comparison3_Samplewise_Outputs/B10BR_HIL/specificity_analysis.csv")
out_path_balbc = Path(f"Comparison3_Samplewise_Outputs/BALBc_HIL/specificity_analysis.csv")

specificity_b10br.drop(columns=["prop_vector"]).to_csv(out_path_b10br, index=False)
specificity_balbc.drop(columns=["prop_vector"]).to_csv(out_path_balbc, index=False)

print(f"Saved: {out_path_b10br}")
print(f"N clonotypes (≥5 cells): {len(specificity_b10br)}")
print("\nSpecificity breakdown:")
print(specificity_b10br["classification"].value_counts())
print("\n\nNoisiest peptides:")
print(specificity_b10br["most_disruptive_peptide"].value_counts().head(10))
print("\n\nCoherentest peptides:")
print(specificity_b10br["most_stabilizing_peptide"].value_counts().head(10))

print(f"Saved: {out_path_balbc}")
print(f"N clonotypes (≥5 cells): {len(specificity_balbc)}")
print("\nSpecificity breakdown:")
print(specificity_balbc["classification"].value_counts())
print("\n\nNoisiest peptides:")
print(specificity_balbc["most_disruptive_peptide"].value_counts().head(10))
print("\n\nCoherentest peptides:")
print(specificity_balbc["most_stabilizing_peptide"].value_counts().head(10))

Saved: Comparison3_Samplewise_Outputs/B10BR_HIL/specificity_analysis.csv
N clonotypes (≥5 cells): 461

Specificity breakdown:
classification
multiple    185
single      164
dual        112
Name: count, dtype: int64


Noisiest peptides:
most_disruptive_peptide
RTYTYEKL    102
VGPRYTNL     69
RAYLFNSV     66
SNYLFTKL     44
VIVRFLTV     41
INFDFPKL     19
SVYVYKVL     18
VAFDFTKV      7
ATLVFHNL      4
HIYEFPQL      3
Name: count, dtype: int64


Coherentest peptides:
most_stabilizing_peptide
RAYLFNSV    71
RTYTYEKL    68
SNYLFTKL    57
VGPRYTNL    52
VIVRFLTV    35
VSFTYRYL    21
INFDFPKL    16
SVYVYKVL    15
SSYTFPKM    12
ATLVFHNL    12
Name: count, dtype: int64
Saved: Comparison3_Samplewise_Outputs/BALBc_HIL/specificity_analysis.csv
N clonotypes (≥5 cells): 241

Specificity breakdown:
classification
multiple    166
single       45
dual         30
Name: count, dtype: int64


Noisiest peptides:
most_disruptive_peptide
VIVRFLTV    66
RTYTYEKL    32
VGPRYTNL    29
SNYLFTKL    26
RAYLFNSV 

In [105]:
def plot_clonotypes_with_specificity_highlights_pdf(
    df: pd.DataFrame,
    spec_df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    out_pdf: Path,
    title_prefix: str = "",
    min_size: int = 5,
    normalize: bool = True,
    jitter_width: float = 0.25,
    plots_per_page: int = 12,
):
    """
    Multi-page PDF showing per-cell binding distributions for each clonotype,
    with specific peptides highlighted.
    
    Ordered by: classification (single → dual → multiple), then by coherence (descending).
    
    Parameters
    ----------
    df : pd.DataFrame
        Cell-level data (after filtering)
    spec_df : pd.DataFrame
        Output from full_specificity_analysis() containing classification and top_peptides
    ct_col : str
        Clonotype column name
    markers : list[str]
        Dextramer marker column names
    peptides : list[str]
        Short peptide names for x-axis labels
    out_pdf : Path
        Output PDF path
    title_prefix : str
        Prefix for page titles
    min_size : int
        Minimum cells per clonotype
    normalize : bool
        If True, plot proportions. If False, plot raw counts.
    jitter_width : float
        Horizontal jitter for dots
    plots_per_page : int
        Number of panels per page
    """
    out_pdf = Path(out_pdf)
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    
    class_order = {"single": 0, "dual": 1, "multiple": 2}
    spec_df = spec_df.copy()
    spec_df["_class_order"] = spec_df["classification"].map(class_order)
    spec_df = spec_df.sort_values(
        ["_class_order", "coherence_full"], 
        ascending=[True, False]
    ).reset_index(drop=True)
    
    n_clonotypes = len(spec_df)
    print(f"Plotting {n_clonotypes} clonotypes with specificity highlights")
    
    if plots_per_page == 12:
        nrows, ncols = 3, 4
    elif plots_per_page == 16:
        nrows, ncols = 4, 4
    elif plots_per_page == 9:
        nrows, ncols = 3, 3
    else:
        ncols = 4
        nrows = (plots_per_page + ncols - 1) // ncols
    
    n_pages = (n_clonotypes + plots_per_page - 1) // plots_per_page
    
    ylabel = "Proportion" if normalize else "Raw count"
    xs = np.arange(len(peptides))
    
    highlight_color = "crimson"
    normal_color = "steelblue"
    mean_marker_highlight = "darkred"
    mean_marker_normal = "black"
    
    with PdfPages(out_pdf) as pdf:
        for page_idx in range(n_pages):
            start = page_idx * plots_per_page
            end = min(start + plots_per_page, n_clonotypes)
            chunk = spec_df.iloc[start:end]
            
            fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
            axes = axes.flatten()
            
            for i, (_, row) in enumerate(chunk.iterrows()):
                ax = axes[i]
                ct = row["clonotype"]
                classification = row["classification"]
                coherence = row["coherence_full"]
                top_peptides = row["top_peptides"]
                cumulative_prop = row["cumulative_proportion"]
                
                df_ct = df[df[ct_col] == ct]
                X = df_ct[markers].to_numpy(dtype=float)
                n_cells = X.shape[0]
                
                if normalize:
                    row_sums = X.sum(axis=1, keepdims=True)
                    row_sums[row_sums == 0] = 1.0
                    X = X / row_sums
                
                highlight_mask = [p in top_peptides for p in peptides]
                
                for j in range(n_cells):
                    jitter = np.random.uniform(-jitter_width, jitter_width, len(peptides))
                    colors = [highlight_color if highlight_mask[k] else normal_color for k in range(len(peptides))]
                    ax.scatter(xs + jitter, X[j, :], alpha=0.5, s=15, c=colors, edgecolors="none")
                
                mean_vals = X.mean(axis=0)
                for k in range(len(peptides)):
                    color = mean_marker_highlight if highlight_mask[k] else mean_marker_normal
                    ax.scatter(xs[k], mean_vals[k], marker="D", s=50, color=color, zorder=10)
                
                ax.set_xticks(xs)
                xticklabels = ax.set_xticklabels(peptides, rotation=90, fontsize=6)
                for k, label in enumerate(xticklabels):
                    if highlight_mask[k]:
                        label.set_color(highlight_color)
                        label.set_fontweight("bold")
                
                ax.tick_params(axis="y", labelsize=7)
                ax.set_ylabel(ylabel, fontsize=7)
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)
                
                rank = start + i + 1
                top_pep_str = "+".join(top_peptides[:2])
                if len(top_peptides) > 2:
                    top_pep_str += f"+{len(top_peptides)-2}more"
                
                title_color = {"single": "darkgreen", "dual": "darkorange", "multiple": "darkred"}
                ax.set_title(
                    f"#{rank} {ct}\n{classification.upper()} ({top_pep_str})\ncoherence={coherence:.3f}, n={n_cells}, cumulative sum={cumulative_prop:.2f}",
                    fontsize=7,
                    color=title_color.get(classification, "black")
                )
            
            for j in range(len(chunk), len(axes)):
                axes[j].axis("off")
            
            fig.suptitle(
                f"{title_prefix} | Clonotypes by specificity (page {page_idx + 1}/{n_pages})",
                fontsize=12, y=0.995
            )
            fig.tight_layout(rect=[0, 0, 1, 0.97])
            pdf.savefig(fig)
            plt.close(fig)
    
    print(f"Saved: {out_pdf}")

In [106]:
plot_clonotypes_with_specificity_highlights_pdf(
    df=df_b10br_combined,
    spec_df=specificity_b10br,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    out_pdf=Path("Comparison3_Samplewise_Outputs/B10BR_HIL/counts_clonotypes_specificity_highlights.pdf"),
    title_prefix="B10BR_HIL Combined",
    min_size=5,
    normalize=False,
    plots_per_page=12,
)

plot_clonotypes_with_specificity_highlights_pdf(
    df=df_balbc_combined,
    spec_df=specificity_balbc,
    ct_col=CT_COL,
    markers=markers_balbc,
    peptides=peptides_balbc,
    out_pdf=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/counts_clonotypes_specificity_highlights.pdf"),
    title_prefix="BALBc_HIL Combined",
    min_size=5,
    normalize=False,
    plots_per_page=12,
)

plot_clonotypes_with_specificity_highlights_pdf(
    df=df_b10br_combined,
    spec_df=specificity_b10br,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    out_pdf=Path("Comparison3_Samplewise_Outputs/B10BR_HIL/prop_clonotypes_specificity_highlights.pdf"),
    title_prefix="B10BR_HIL Combined",
    min_size=5,
    normalize=True,
    plots_per_page=12,
)

plot_clonotypes_with_specificity_highlights_pdf(
    df=df_balbc_combined,
    spec_df=specificity_balbc,
    ct_col=CT_COL,
    markers=markers_balbc,
    peptides=peptides_balbc,
    out_pdf=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/prop_clonotypes_specificity_highlights.pdf"),
    title_prefix="BALBc_HIL Combined",
    min_size=5,
    normalize=True,
    plots_per_page=12,
)

Plotting 461 clonotypes with specificity highlights
Saved: Comparison3_Samplewise_Outputs/B10BR_HIL/counts_clonotypes_specificity_highlights.pdf
Plotting 241 clonotypes with specificity highlights
Saved: Comparison3_Samplewise_Outputs/BALBc_HIL/counts_clonotypes_specificity_highlights.pdf
Plotting 461 clonotypes with specificity highlights
Saved: Comparison3_Samplewise_Outputs/B10BR_HIL/prop_clonotypes_specificity_highlights.pdf
Plotting 241 clonotypes with specificity highlights
Saved: Comparison3_Samplewise_Outputs/BALBc_HIL/prop_clonotypes_specificity_highlights.pdf
